# Evaluación cuantitativa: modelo afinado en Vertex AI vs. modelo base

Igual que `evaluate_gemma.ipynb`, pero para los adaptadores LoRA entrenados
con un **Custom Training Job** de Vertex AI / Gemini Enterprise Agent
Platform (*Agent Platform -> Models -> Training*), cuyo resultado quedó en
un bucket de Cloud Storage en vez del disco de esta VM.

Usamos el split **`test`** de `knkarthick/samsum` (resúmenes escritos por
humanos, que el modelo **no** vio durante el entrenamiento) para comparar el
modelo base contra el afinado con:

- **ROUGE-1 / ROUGE-2 / ROUGE-L / ROUGE-Lsum** — solapamiento de n-gramas
  contra la referencia.
- **BERTScore F1** — similitud semántica vía embeddings; reconoce
  paráfrasis que ROUGE no ve.
- **Longitud promedio** del resumen generado, comparada con la referencia
  humana (samsum pide 1-2 frases).

## Instalar dependencias adicionales


In [ ]:
%pip install --quiet evaluate rouge_score bert_score absl-py pandas google-cloud-storage


## 0. Configuración

Cambia `GCS_ADAPTER_DIR` por la ruta real del bucket donde quedó el
resultado del Custom Training Job (consola de Agent Platform -> Models ->
Training, detalles del job, "output directory").


In [ ]:
import os

MODEL_NAME = "google/gemma-7b-it"
GCS_ADAPTER_DIR = "gs://[tu-bucket]/gemma-7b-it-samsum-lora"   # <-- reemplaza esto
LOCAL_CACHE_DIR = "/home/jovyan/labs/gemma-7b-it-samsum-lora-vertex"
FORCE_DOWNLOAD = False

DATASET_NAME = "knkarthick/samsum"
EVAL_SPLIT = "test"          # split de PRUEBA -no visto en el entrenamiento
N_EXAMPLES = 30               # empieza chico para iterar rápido; sube esto para un número más confiable
SEED = 42                     # para que la muestra sea reproducible
MAX_NEW_TOKENS = 64
BERTSCORE_MODEL_TYPE = "distilbert-base-uncased"  # chico (268MB); "roberta-large" (~1.4GB) es más preciso
OUTPUT_CSV = os.environ.get("EVAL_OUTPUT_CSV", "/home/jovyan/labs/eval_base_vs_finetuned_vertex.csv")

assert GCS_ADAPTER_DIR.startswith("gs://"), "GCS_ADAPTER_DIR debe empezar con 'gs://'"


## 1. Descargar los adaptadores desde Cloud Storage (con caché local)

In [ ]:
def download_gcs_dir(gcs_uri, local_dir, force=False):
    if os.path.isdir(local_dir) and os.listdir(local_dir) and not force:
        print(f"Ya existe una copia local en {local_dir}. Saltando descarga.")
        return local_dir

    from google.cloud import storage

    bucket_name, _, prefix = gcs_uri[len("gs://"):].partition("/")
    prefix = prefix.rstrip("/")

    print(f"Descargando {gcs_uri} -> {local_dir} ...")
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blobs = [b for b in bucket.list_blobs(prefix=prefix + "/" if prefix else prefix)
             if not b.name.endswith("/")]

    if not blobs:
        raise SystemExit(f"No encontré archivos en {gcs_uri}. ¿Es correcta la ruta del bucket?")

    os.makedirs(local_dir, exist_ok=True)
    for blob in blobs:
        rel_path = blob.name[len(prefix):].lstrip("/") if prefix else blob.name
        dest = os.path.join(local_dir, rel_path)
        os.makedirs(os.path.dirname(dest) or local_dir, exist_ok=True)
        blob.download_to_filename(dest)

    print(f"Descarga completa: {len(blobs)} archivo(s) en {local_dir}")
    return local_dir


ADAPTER_DIR = download_gcs_dir(GCS_ADAPTER_DIR, LOCAL_CACHE_DIR, force=FORCE_DOWNLOAD)
print("Adaptadores disponibles en:", ADAPTER_DIR)


## 2. Cargar el modelo base (4-bit) + los adaptadores LoRA

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("CUDA disponible:", torch.cuda.is_available())

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Modelo + adaptadores (entrenados en Vertex AI) cargados.")


## 3. Funciones de generación (idénticas a `infer_gemma_vertex.ipynb`)

In [ ]:
def build_prompt(dialogue):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{dialogue}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def resumir(dialogue, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(dialogue)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def resumir_base(dialogue, max_new_tokens=MAX_NEW_TOKENS):
    with model.disable_adapter():
        return resumir(dialogue, max_new_tokens)


## 4. Tomar una muestra del split de prueba y generar con ambos modelos

Esto es lo que toma tiempo: dos generaciones (base + afinado) por cada
ejemplo. Con `N_EXAMPLES=30` en una L4 debería tomar unos minutos.


In [ ]:
from datasets import load_dataset

test_dataset = load_dataset(DATASET_NAME, split=EVAL_SPLIT)
test_dataset = test_dataset.shuffle(seed=SEED).select(range(min(N_EXAMPLES, len(test_dataset))))
print(f"Evaluando sobre {len(test_dataset)} ejemplos de {DATASET_NAME} ({EVAL_SPLIT})")

dialogues, references = [], []
preds_finetuned, preds_base = [], []

for i, example in enumerate(test_dataset):
    dialogues.append(example["dialogue"])
    references.append(example["summary"])

    preds_finetuned.append(resumir(example["dialogue"]))
    preds_base.append(resumir_base(example["dialogue"]))

    if (i + 1) % 10 == 0 or (i + 1) == len(test_dataset):
        print(f"  ... {i + 1}/{len(test_dataset)} ejemplos generados")


## 5. Calcular ROUGE y BERTScore

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

rouge_finetuned = rouge.compute(predictions=preds_finetuned, references=references)
rouge_base = rouge.compute(predictions=preds_base, references=references)

rouge_finetuned_per_example = rouge.compute(
    predictions=preds_finetuned, references=references, use_aggregator=False
)
rouge_base_per_example = rouge.compute(
    predictions=preds_base, references=references, use_aggregator=False
)

print("ROUGE (base):     ", rouge_base)
print("ROUGE (afinado):  ", rouge_finetuned)


In [ ]:
bertscore = evaluate.load("bertscore")

bs_finetuned = bertscore.compute(
    predictions=preds_finetuned, references=references,
    lang="en", model_type=BERTSCORE_MODEL_TYPE,
)
bs_base = bertscore.compute(
    predictions=preds_base, references=references,
    lang="en", model_type=BERTSCORE_MODEL_TYPE,
)

bertscore_f1_base = sum(bs_base["f1"]) / len(bs_base["f1"])
bertscore_f1_finetuned = sum(bs_finetuned["f1"]) / len(bs_finetuned["f1"])

print("BERTScore F1 (base):    ", round(bertscore_f1_base, 4))
print("BERTScore F1 (afinado): ", round(bertscore_f1_finetuned, 4))


## 6. Tabla comparativa

In [ ]:
import pandas as pd

df_resumen = pd.DataFrame({
    "modelo": ["base (sin LoRA)", "afinado (con LoRA, Vertex AI)"],
    "rouge1": [rouge_base["rouge1"], rouge_finetuned["rouge1"]],
    "rouge2": [rouge_base["rouge2"], rouge_finetuned["rouge2"]],
    "rougeL": [rouge_base["rougeL"], rouge_finetuned["rougeL"]],
    "rougeLsum": [rouge_base["rougeLsum"], rouge_finetuned["rougeLsum"]],
    "bertscore_f1": [bertscore_f1_base, bertscore_f1_finetuned],
    "longitud_promedio_palabras": [
        sum(len(p.split()) for p in preds_base) / len(preds_base),
        sum(len(p.split()) for p in preds_finetuned) / len(preds_finetuned),
    ],
})

referencia_len = sum(len(r.split()) for r in references) / len(references)
print(f"(longitud promedio de la referencia humana: {referencia_len:.1f} palabras)")
df_resumen


## 7. Guardar el detalle por ejemplo (para inspección manual)

Las columnas `rougeL_base` / `rougeL_afinado` te dejan ordenar y encontrar
los mejores y peores casos de cada modelo.


In [ ]:
df_detalle = pd.DataFrame({
    "dialogo": dialogues,
    "referencia": references,
    "resumen_base": preds_base,
    "resumen_afinado": preds_finetuned,
    "rougeL_base": rouge_base_per_example["rougeL"],
    "rougeL_afinado": rouge_finetuned_per_example["rougeL"],
})

df_detalle.to_csv(OUTPUT_CSV, index=False)
print(f"Detalle guardado en: {OUTPUT_CSV}")

df_detalle["mejora"] = df_detalle["rougeL_afinado"] - df_detalle["rougeL_base"]
df_detalle.sort_values("mejora", ascending=False).head(3)[
    ["dialogo", "referencia", "resumen_base", "resumen_afinado", "mejora"]
]


## Notas finales

- **¿Por qué comparar contra el split `test` y no `train`?** Porque evaluar
  sobre ejemplos que el modelo ya vio durante el entrenamiento sobreestima
  qué tan bien generaliza.
- **¿Por qué descargar del bucket en vez de montarlo?** Podrías usar
  Cloud Storage FUSE para montar el bucket como si fuera un directorio local,
  pero para un adaptador LoRA (unos pocos MB-cientos de MB) es más simple y
  más rápido bajarlo una vez a disco local con el cliente de Python.
- Si te da un error de permisos (403) al descargar, revisa que la cuenta de
  servicio de esta VM tenga el rol *Storage Object Viewer* sobre el bucket.
- Este mismo pipeline existe como script plano en `evaluate_gemma_vertex.py`
  (con flags `--n_examples`, `--skip_bertscore`, `--gcs_adapter_dir`, etc.).
